# Dynamic circuits in PureMagic: a walkthrough

This notebook follows one tiny circuit through the whole pipeline: from an ordinary circuit, to a
**dynamic circuit** with a mid-circuit measurement and outcome-dependent corrections, to the
lattice-surgery schedule PureMagic produces for it. The steps after Clifford+T are the same as in
`puremagic_tutorial.ipynb`; read that one first for the scheduler itself. This one covers what
changes when the circuit measures and reacts in the middle.

- **dcsynth** (the `DynamicCircuitPlay` Artifact) turns blocks of a circuit into dynamic circuits
  that use an ancilla, a mid-circuit measurement and a correction, so as to lower **CNOT depth**.
- **PureMagic** schedules the result. Measurements become Pauli product measurements, a reset
  becomes a classical frame update, and every operation that depends on an outcome waits for it
  (plus the decoder's reaction latency). Conditional operations run only when their condition
  holds.

The full specification of the mid-circuit-measurement support is `docs/mcm-changes.md`.

## 1. The pipeline

```
circuit.qasm
   |  dcsynth compile_blocks   (Artifact .venv)  partition into 3-qubit blocks; synthesize a
   |                                               dynamic version of each (ancilla + measurement)
   v
block library                 (precompiled for this demo: about 8 minutes to rebuild)
   |  dcsynth assemble        (Artifact .venv, notebooks/dcsynth_assemble.py)
   v
dynamic.qpy                   measure / reset / if_else corrections
   |  dcsynth_to_cliffordt.py (PureMagic .venv)  Clifford+T, within a total error budget
   v
dynamic.cliffordt.qasm        `if(m0==1) t q[1];` lines
   |  transpile               Pauli products; conditions become `@` (wait for) and `?` (only if)
   v
dynamic.trans + .trans.dyn.json
   |  puremagic               schedule: reaction latency, sampled outcomes, hosted ancillas
   v
dynamic.schedule  ->  plot_dynamic.py (independent check + timeline)
```

Two Python environments are involved: dcsynth runs in the Artifact's `.venv` (Python 3.10,
BQSKit), everything else in PureMagic's. The notebook runs in PureMagic's environment and calls
dcsynth as a subprocess.

In [ ]:
%matplotlib inline

import json
import os
import pathlib
import re
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start="."):
    """Walk upward from `start` looking for the PureMagic repo root."""
    p = pathlib.Path(start).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "Cargo.toml").exists() and (candidate / "data_processing").is_dir():
            return candidate
    raise RuntimeError("Could not locate the PureMagic repo root (looked for Cargo.toml)")


REPO_ROOT = find_repo_root(".")
BIN = REPO_ROOT / "target" / "release"
for exe in ["transpile", "puremagic"]:
    if not (BIN / exe).exists():
        raise RuntimeError(f"{exe} not found under {BIN} -- run `cargo build --release` first")

# dcsynth lives in the DynamicCircuitPlay Artifact, with its own Python environment. Point
# DCSYNTH_ARTIFACT at it if it is not next to this repository.
ARTIFACT = pathlib.Path(os.environ.get(
    "DCSYNTH_ARTIFACT", REPO_ROOT.parent / "DynamicCircuitPlay" / "Artifact")).resolve()
ARTIFACT_PY = ARTIFACT / ".venv" / "bin" / "python"
if not ARTIFACT_PY.exists():
    raise RuntimeError(f"dcsynth environment not found at {ARTIFACT_PY}; set DCSYNTH_ARTIFACT")

sys.path.insert(0, str(REPO_ROOT / "data_processing"))
import dcsynth_to_cliffordt as lowering  # noqa: E402
from qiskit import qpy  # noqa: E402

DEMO = REPO_ROOT / "notebooks" / "data" / "dynamic_demo"
work_dir = pathlib.Path(tempfile.mkdtemp(prefix="dynamic_tutorial_"))
print("Repo root:", REPO_ROOT)
print("dcsynth Artifact:", ARTIFACT)
print("Scratch working directory:", work_dir)


def render_terminal_output(text):
    """Replay '\r'/'\b' so captured stdout looks the way it would on a terminal."""
    lines, buf, cursor = [], [], 0
    for ch in text:
        if ch == "\n":
            lines.append("".join(buf))
            buf, cursor = [], 0
        elif ch == "\r":
            cursor = 0
        elif ch == "\x08":
            cursor = max(0, cursor - 1)
        else:
            if cursor < len(buf):
                buf[cursor] = ch
            else:
                buf.append(ch)
            cursor += 1
    lines.append("".join(buf))
    return re.sub(r"\x1b\[[0-9;]*m", "", "\n".join(lines))


def run(cmd, quiet=False, cwd=None, **kwargs):
    """Run a command (in work_dir by default), print it and its output; raise on failure."""
    printable = " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    result = subprocess.run(cmd, cwd=cwd or work_dir, capture_output=True, text=True, **kwargs)
    if not quiet:
        print(render_terminal_output(result.stdout + result.stderr).strip())
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"command failed ({result.returncode}): {printable}")
    return result


def report(stdout, keys=("Scheduled", "Conditional products", "Hosted ancilla", "T gate failures",
                         "Parallelism", "Volume estimate", "efficiency")):
    """The lines of a puremagic report worth reading."""
    out = render_terminal_output(stdout)
    return "\n".join(l for l in out.splitlines() if any(k in l for k in keys))

## 2. The circuit

Three qubits, each pair coupled by a ZZ interaction (CX, Rz, CX), between a layer of H and a layer of
Rx: one QAOA-style step on a triangle. Its CNOT depth is 6. dcsynth partitions circuits into
3-qubit blocks, so this whole circuit is **one block**.

In [ ]:
circuit = work_dir / "demo.qasm"
shutil.copy(DEMO / "demo.qasm", circuit)
print(circuit.read_text())

## 3. dcsynth: compile the block, then assemble a dynamic circuit

`compile_blocks.py` partitions the circuit and, per block, keeps three forms: the block as
partitioned (`ori`), BQSKit's resynthesis (`bq`), and **dynamic versions** (`dc`). A dynamic
version is a main circuit on the block's qubits plus an ancilla, a measurement of the ancilla,
and a single-qubit correction layer for each outcome (`branch0`, `branch1`). dcsynth searches for
the shallowest such circuit whose infidelity against the block is below `--epsilon`.

The search takes about 8 minutes for this one block, so a precompiled library ships in
`notebooks/data/dynamic_demo/`. Set `COMPILE_FRESH = True` to rebuild it.

In [ ]:
COMPILE_FRESH = False

lib = work_dir / "exp" / "compilation" / "demo@nb"
if COMPILE_FRESH:
    (work_dir / "exp" / "circuit").mkdir(parents=True)
    shutil.copy(circuit, work_dir / "exp" / "circuit" / "demo.qasm")
    run([ARTIFACT_PY, ARTIFACT / "scripts" / "compile_blocks.py", "-b", "demo",
         "--exp-dir", work_dir / "exp", "--tag", "nb", "--epsilon", "1e-6", "--backend", "jax"],
        quiet=True, cwd=ARTIFACT)
else:
    shutil.copytree(DEMO / "demo@nb", lib)

for form in ("ori", "bq", "dc"):
    print(f"{form:3}  CNOT depth:", (lib / f"depth_info_{form}.csv").read_text().splitlines()[1].split(",")[1])
print("\nfiles:", sorted(p.name for p in lib.iterdir()))

`dcsynth_assemble.py` (run in dcsynth's environment) picks, per block, whichever form gives the
smallest CNOT depth, on the fewest ancillas that reach it (dcsynth's own `fewest_ancillas` rule).
It removes any declared ancilla no gate touches, since PureMagic gives every qubit a patch. It
also computes the block's **realized synthesis error** from its actual circuits. dcsynth's
`--epsilon` bounds an *infidelity* (a squared quantity), so the error, a distance, is roughly its
square root.

In [ ]:
assembled = work_dir / "assembled"
run([ARTIFACT_PY, REPO_ROOT / "notebooks" / "dcsynth_assemble.py", circuit, lib, assembled],
    quiet=True, cwd=ARTIFACT)
summary = json.loads((assembled / "summary.json").read_text())
print(json.dumps(summary, indent=1))

orig_qc = qpy.load(open(assembled / "orig.qpy", "rb"))[0]
dyn_qc = qpy.load(open(assembled / "dynamic.qpy", "rb"))[0]

In [ ]:
print(orig_qc.draw("text", fold=110))

In [ ]:
print(dyn_qc.draw("text", fold=110))

The dynamic circuit has one extra qubit (`q_3`, the ancilla). Its main circuit runs two CNOT
layers in parallel on the ancilla, so the CNOT depth drops from 6 to 4. The ancilla is then
measured into `c_0` and reset (`|0>`). The data qubits get an `If-0 / Else-0` correction: one
single-qubit unitary if the outcome was 1, another if it was 0.

## 4. Clifford+T, within a total error budget

`dcsynth_to_cliffordt.py` turns every single-qubit unitary into Clifford+T with cyclosynth and
writes the restricted QASM dialect PureMagic's `transpile` reads (`if(m0==1) t q[1];`). For each
correction it applies one branch's unitary unconditionally and conditions only the **difference**
between the branches. If the difference is a Pauli, it costs nothing: it becomes a classical
frame update.

**Error budget.** All errors are measured in one unit, half the diamond distance from the ideal
operation, and that unit adds up over a circuit. So one total budget can be shared between:

- dcsynth's block (fixed: computed above);
- every cyclosynth rotation (each within ε);
- every Clifford snap (each within the snap threshold τ).

Given a budget, the lowering gives every rotation the same ε (this minimizes T count) and snaps any
target within τ = ε(1 + ln(1/ε)) of a Clifford (snapping beats synthesizing exactly then). The
derivation is in `docs/mcm-changes.md`, section 6a.

In [ ]:
BUDGET = 1e-3
rows = []
for name, qc, block_error in (("orig", orig_qc, 0.0), ("dynamic", dyn_qc, summary["block_error"])):
    text, st = lowering.lower(qc, error_budget=BUDGET, block_error=block_error)
    (work_dir / f"{name}.cliffordt.qasm").write_text(text)
    rows.append({"circuit": name, "T": st["t"], "conditional T": st["conditional_t"],
                 "eps": st["epsilon"], "snap tol": st["clifford_tol"],
                 "rotations": st["n_rotations"], "snaps": st["n_snaps"],
                 "block error": st["error_blocks"], "lowering error": st["error_lowering"],
                 "total error": st["error_total"]})
pd.DataFrame(rows).set_index("circuit")

The two circuits are not equally expensive: dcsynth's main circuit is built from generic
rotations, so the dynamic version needs several times the T gates of the original, whose rotations
are few. This is the main cost of synthesized dynamic circuits in a fault-tolerant setting.

The conditional part of the dynamic circuit (the corrections):

In [ ]:
dyn_lines = (work_dir / "dynamic.cliffordt.qasm").read_text().splitlines()
print("\n".join(l for l in dyn_lines if l.startswith(("measure", "reset", "creg"))))
cond = [l for l in dyn_lines if l.startswith("if(")]
print(f"\n{len(cond)} conditional gates, e.g.:")
print("\n".join(cond[:8]))

How the T count depends on the budget (only the lowering is redone; the block error stays fixed):

In [ ]:
sweep = []
for budget in (2e-4, 5e-4, 1e-3, 1e-2, 1e-1):
    for name, qc, be in (("orig", orig_qc, 0.0), ("dynamic", dyn_qc, summary["block_error"])):
        _, st = lowering.lower(qc, error_budget=budget, block_error=be)
        sweep.append({"budget": budget, "circuit": name, "T": st["t"] + st["conditional_t"]})
sweep = pd.DataFrame(sweep)
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, g in sweep.groupby("circuit"):
    ax.plot(g["budget"], g["T"], marker="o", label=name)
ax.set_xscale("log")
ax.set_xlabel("total error budget (half diamond distance)")
ax.set_ylabel("T count")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5. Transpile to Pauli products

As in the static tutorial, `transpile` pushes every Clifford to the end (Heisenberg picture), so
T gates become π/8 rotations on multi-qubit Paulis, and measurements become multi-qubit Pauli
measurements. What is new:

| In the input | In `.trans` |
|---|---|
| `measure q[3] -> m0[0];` | a `<M>` product (often multi-qubit) with an outcome id |
| `reset q[3];` | that measurement, plus `X^m` owed to the classical **Pauli frame** (no lattice operation) |
| `if(m0==1) x q[i];` | a frame update only |
| `if(m0==1) t q[i];`, `h`, `s`, `cx` | pending Cliffords are flushed, the frame on the qubit is paid off, and the gate is emitted with **`@k`** (wait for product k's outcome) and **`?k=1`** (run only if it is 1) |

`--reset_mode flush` flushes pending Cliffords at each reset, so the ancilla's measurement is a
single-patch measurement. That is also what lets a hosted ancilla patch go back to cultivating
(section 6).

In [ ]:
for name in ("orig", "dynamic"):
    run([BIN / "transpile", "-i", f"{name}.cliffordt.qasm", "-o", f"{name}.trans",
         "--defer_trailing", "--reset_mode", "flush"], quiet=True)
trans = (work_dir / "dynamic.trans").read_text().splitlines()
kinds = pd.Series([re.search(r"<(\w+)>", l).group(1) for l in trans]).value_counts()
print("dynamic.trans:", len(trans), "products:", dict(kinds))
meas = [i for i, l in enumerate(trans) if "<M>" in l]
print("\nthe mid-circuit measurement and the first conditional products:")
first = meas[0]
for i, l in enumerate(trans):
    if i == first or ("@" in l and len([x for x in trans[:i] if "@" in x]) < 6):
        print(f"{i:5}: {l}")

The sidecar `dynamic.trans.dyn.json` carries the classical meaning that the scheduler does not
need but an executor would: each product's sign and which outcomes flip it (`frame_deps`), each
measurement's outcome id, and each conditional product's condition (`cond`, the same as `?` but in
outcome ids):

In [ ]:
side = json.loads((work_dir / "dynamic.trans.dyn.json").read_text())
recs = side["products"]
print(json.dumps([r for r in recs if r.get("meas_id") is not None][:2], indent=1))
print(json.dumps([r for r in recs if r.get("cond")][:2], indent=1))

**Exact check.** For circuits this small, `validate_dynamic.py` simulates the Clifford+T input and
the `.trans` file (with its sidecar) exactly, branching at every measurement. It compares their
joint outcome distributions in every reset mode. (It is skipped above 10 qubits; for larger
circuits only the schedule checks in section 6 run.)

In [ ]:
res = run([sys.executable, REPO_ROOT / "data_processing" / "validate_dynamic.py",
           "dynamic.cliffordt.qasm"], quiet=True)
print(res.stdout.strip().splitlines()[-1])

## 6. Schedule with PureMagic

The flags that matter for dynamic circuits (the rest are as in `puremagic_tutorial.ipynb`):

| Flag | Default | Effect |
|---|---|---|
| `--reaction-latency L` | 0 | A product that needs an outcome (`@`) starts at least L lcycles after the measurement ends: the time to decode it |
| `--cond-outcomes random\|worst` | random | `random` samples each measurement's outcome and **skips** conditional products whose condition fails; `worst` runs them all (an upper bound) |
| `--cond-p1 P` | 0.5 | Probability that a measurement's outcome is 1, for `random` |
| `--ancilla-qubits a,b` | none | Host these qubits on injection patches: they cultivate magic states while idle (needs `-u`) |

PureMagic is an online scheduler: each lcycle it picks among the products that are ready,
given the magic states available. Outcomes are drawn only when a product that depends on them
becomes ready, so no decision uses information the hardware would not have yet.

In [ ]:
L = 5
SEED = "6"  # with this seed, the mid-circuit measurement samples outcome 0
ANCILLA = str(summary["data_qubits"])  # the dynamic circuit's ancilla is the qubit after the data
# Each run writes <circuit>.schedule; the plain "dynamic" run goes last, so dynamic.schedule (checked
# and plotted in section 7) is that run's.
runs = {
    "orig": ["orig.trans"],
    "dynamic, worst case": ["dynamic.trans", "--reaction-latency", str(L), "--cond-outcomes", "worst"],
    "dynamic, hosted ancilla": ["dynamic.trans", "--reaction-latency", str(L),
                                "--ancilla-qubits", ANCILLA],
    "dynamic": ["dynamic.trans", "--reaction-latency", str(L)],
}
outputs = {}
for label, args in runs.items():
    res = run([BIN / "puremagic", "-u", "-r", SEED, "-c", *args]
              + (["--plot", "topo,paths"] if label == "dynamic" else []), quiet=True)
    outputs[label] = res.stdout
    print(f"--- {label}\n{report(res.stdout)}\n")

Reading the new report lines:

- **Conditional products: N, skipped K on M sampled outcomes**: the measurement's outcome was
  sampled, and the K products that the other outcome would need were skipped (no lcycle, no magic
  state, no routing). With this seed the outcome is 0. Every correction here reads `?…=1`, because
  the lowering applied the outcome-0 branch unconditionally and conditioned only the difference.
  So all of them are skipped.
- **Hosted ancilla qN: live X / Y lcycles**: with `--ancilla-qubits`, the ancilla's patch is a
  data patch only while it is in use, and cultivates magic states otherwise.
- The worst-case run executes every conditional product.

The topology is laid out for 4 data qubits (the ancilla is a data patch unless hosted):

In [ ]:
display(Image(filename=str(work_dir / "dynamic.topo.png")))

In [ ]:
paths = sorted((work_dir / "dynamic.paths").glob("*.png"), key=lambda p: int(p.stem.split(".")[-1]))
display(Image(filename=str(paths[min(3, len(paths) - 1)])))

## 7. Checking the schedule

`plot_dynamic.py` re-derives every constraint from the `.trans` file and checks the schedule
against them, independently of the scheduler. It checks:

- every product ran once, or was skipped because its logged condition failed;
- qubit order is respected (counting through skipped products);
- every `@` product started at least L+1 lcycles after its measurement;
- no qubit was used twice in an lcycle.

It also draws a timeline: conditional products are outlined in red, with an arrow from the
measurement they waited for.

In [ ]:
res = run([sys.executable, REPO_ROOT / "data_processing" / "plot_dynamic.py", "dynamic.trans",
           "-o", "dynamic_check"], quiet=True)
print(res.stdout.strip())
display(Image(filename=str(work_dir / "dynamic_check.timeline.png")))

The long grey run on `q2` near the end is about 180 *physical* single-qubit Cliffords (S/SX,
3 lcycles each, in series). A static circuit has none: `transpile` pushes every Clifford to the
end. But before each physical conditional gate it must flush the pending Cliffords, because a
conditional Clifford cannot be absorbed. Removing the conditional lines from this circuit leaves
only 8 of them on `q2`. Avoiding those flushes is the planned "Level 2" work in
`docs/mcm-changes.md`.

The outcomes the scheduler sampled and the products it skipped are logged in the schedule's
header:

In [ ]:
sched_head = [l for l in (work_dir / "dynamic.schedule").read_text().splitlines() if l.startswith("#")]
print("\n".join(l for l in sched_head if l.startswith(("# cond_outcomes", "# outcomes", "# skipped"))))

## 8. Comparing the two circuits

The same statistics as the static tutorial, side by side. Volume is lcycles × total qubits
(patches) on the layout.

In [ ]:
def stats(stdout):
    s = render_terminal_output(stdout)
    m = re.search(r"Scheduled (\d+) in (\d+) logical cycles, volume (\d+)", s)
    g = lambda pat: (re.search(pat, s) or [None, None])[1]
    return {"products run": int(m[1]), "lcycles": int(m[2]), "volume": int(m[3]),
            "parallelism": float(g(r"Parallelism: ([\d.]+)x")),
            "efficiency": float(g(r"Normalized scheduling efficiency: ([\d.]+)")),
            "skipped": int(g(r"skipped (\d+)") or 0)}

table = pd.DataFrame({label: stats(out) for label, out in outputs.items()}).T
table.insert(0, "CNOT depth", [summary["orig"]["cnot_depth"]] + [summary["dynamic"]["cnot_depth"]] * 3)
table.insert(1, "T", [rows[0]["T"]] + [rows[1]["T"] + rows[1]["conditional T"]] * 3)
table

The dynamic circuit is shallower in CNOTs, but costs more spacetime volume, because its
synthesized rotations need many more T gates and PureMagic's volume is driven by magic-state
supply. Skipping the branch that did not happen, and hosting the ancilla on an injection patch,
each recover some of it.

One run shows one outcome. Averaged over seeds, `random` gives the expected cost (here each
outcome comes up half the time), against the `worst` bound:

In [ ]:
avg = []
for seed in range(1, 9):
    for mode in ("random", "worst"):
        res = run([BIN / "puremagic", "-u", "-r", str(seed), "-c", "dynamic.trans",
                   "--reaction-latency", str(L), "--cond-outcomes", mode], quiet=True)
        avg.append({"seed": seed, "mode": mode, **stats(res.stdout)})
avg = pd.DataFrame(avg)
avg.groupby("mode")[["lcycles", "volume", "skipped"]].mean().round(0)

**Reaction latency.** How much the decoder's latency costs depends on how much outcome-dependent
work is on the critical path:

In [ ]:
lat = []
for L_ in (0, 2, 5, 10, 20):
    for label, extra in (("data-patch ancilla", []), ("hosted ancilla", ["--ancilla-qubits", ANCILLA])):
        res = run([BIN / "puremagic", "-u", "-r", SEED, "-c", "dynamic.trans",
                   "--reaction-latency", str(L_), *extra], quiet=True)
        lat.append({"latency": L_, "ancilla": label, **stats(res.stdout)})
lat = pd.DataFrame(lat)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
for label, g in lat.groupby("ancilla"):
    a1.plot(g["latency"], g["lcycles"], marker="o", label=label)
    a2.plot(g["latency"], g["volume"], marker="o", label=label)
a1.set_xlabel("reaction latency (lcycles)"); a1.set_ylabel("lcycles"); a1.legend(); a1.grid(alpha=0.3)
a2.set_xlabel("reaction latency (lcycles)"); a2.set_ylabel("volume"); a2.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Further reading

- `docs/mcm-changes.md`: the specification. It covers the model, the file formats (`@`, `?`,
  the sidecar), the timing model, the error budget (section 6a), verification, costs and
  limitations, and planned work.
- `docs/dynamic-circuits.md`: the user guide for dynamic circuits.
- `puremagic_tutorial.ipynb`: the scheduler itself, its plots and parameters.

From here: try `--cond-p1` to skew the outcome statistics, `--reset_mode frame` in `transpile`
(cheaper, but a hosted ancilla can no longer go dormant), or a larger budget in section 4.